# Grad-CAM gallery

Samples random HAM10000 images and shows, in a table, each original picture next
to its Grad-CAM overlay for the predicted class (or the true class when
`TARGET = "true"`). Masks are used only through the IoU reported per row.

Model: `sgonzalez2000/dermai-efficientnet-b0` (Apache-2.0), 7 HAM10000 classes.

In [ ]:
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "dermai_xai").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from dermai_xai.gradcam import GradCAM
from dermai_xai.localization import iou, threshold_heatmap
from dermai_xai.model import DEFAULT_MODEL_ID, HAM10000_CLASSES, load_model, load_processor
from scripts.evaluate import load_frame
print("repo root:", REPO_ROOT)

## Parameters

In [ ]:
DATA_DIR = Path(r"C:\Users\Amine\skin-cancer-xai-research\reference\dermai-explainability\data")
MASKS_DIR = None  # default: DATA_DIR / "HAM10000_segmentations_lesion_tschandl"
COUNT = 10
SEED = 7
ALPHA = 0.45
TARGET = "predicted"  # "predicted" or "true"
POSITIVE_FRACTION = 0.15
GALLERY_PATH = Path("outputs/gradcam_gallery_notebook.png")

LABEL_TO_INDEX = {name: index for index, name in enumerate(HAM10000_CLASSES)}

## Data and model

In [ ]:
frame, mapping = load_frame(DATA_DIR)
mask_dir = Path(MASKS_DIR) if MASKS_DIR else DATA_DIR / "HAM10000_segmentations_lesion_tschandl"
sampled = frame.sample(min(COUNT, len(frame)), random_state=SEED).reset_index(drop=True)

model = load_model(DEFAULT_MODEL_ID)
processor = load_processor(DEFAULT_MODEL_ID)
cam_engine = GradCAM(model)
colormap = plt.get_cmap("turbo")
print("images:", len(frame), "sampled:", len(sampled))

## Predictions, Grad-CAM and per-image records

In [ ]:
gallery = []
records = []
for _, row in sampled.iterrows():
    image = Image.open(mapping[row.image_id]).convert("RGB").resize((224, 224))
    batch = processor(image, return_tensors="pt")["pixel_values"].to(next(model.parameters()).device)
    with torch.no_grad():
        probabilities = torch.softmax(model(pixel_values=batch).logits, dim=-1)
    predicted = int(probabilities.argmax(dim=-1))

    target_label = predicted if TARGET == "predicted" else LABEL_TO_INDEX[row.dx]
    explanation = cam_engine(batch, torch.tensor([target_label], device=batch.device))
    cam = explanation["cam"][0].detach().float().cpu().numpy()
    heat_rgb = (colormap(cam)[..., :3] * 255).astype(np.uint8)
    overlay = (ALPHA * heat_rgb + (1.0 - ALPHA) * np.asarray(image)).astype(np.uint8)

    mask_path = mask_dir / f"{row.image_id}_segmentation.png"
    localization_text = "no mask"
    if mask_path.is_file():
        truth = np.asarray(Image.open(mask_path).convert("L").resize((224, 224), Image.NEAREST)) > 0
        localization_text = f"IoU {iou(threshold_heatmap(cam, POSITIVE_FRACTION), truth):.2f}"

    confidence = float(probabilities[0, predicted].cpu())
    gallery.append((row.image_id, row.dx, predicted, confidence, localization_text, image, overlay))
    records.append({
        "image_id": row.image_id,
        "true": row.dx,
        "pred": HAM10000_CLASSES[predicted],
        "confidence": round(confidence, 3),
        "localization": localization_text,
    })

pd.DataFrame(records)

## Gallery table

In [ ]:
fig, axes = plt.subplots(len(gallery), 2, figsize=(5.2, 2.7 * len(gallery)))
axes = np.atleast_2d(axes)
for index, (image_id, true_label, predicted, confidence, localization_text, image, overlay) in enumerate(gallery):
    correct = predicted == LABEL_TO_INDEX[true_label]
    label_color = "green" if correct else "red"
    axes[index, 0].imshow(image)
    axes[index, 0].set_title(f"{image_id} | true: {true_label} | pred: {HAM10000_CLASSES[predicted]}", color=label_color, fontsize=9)
    axes[index, 1].imshow(overlay)
    axes[index, 1].set_title(f"Grad-CAM ({TARGET}, p={confidence:.2f}) | {localization_text}", color=label_color, fontsize=9)
    axes[index, 0].axis("off")
    axes[index, 1].axis("off")
fig.tight_layout()
GALLERY_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(GALLERY_PATH, dpi=110, bbox_inches="tight")
print("gallery file:", GALLERY_PATH.resolve())
plt.show()